# Paperspace ComfyUI Runtime

この Notebook は、`/storage/ComfyUI` にある既存の ComfyUI を使い、必要なモデルだけを Hugging Face private repo から `/app/models` に同期して起動するためのものです。


In [ ]:
# Notebook 全体で使う設定値です。
# まずは MODEL_CATEGORY だけ変えれば使える想定です。
from pathlib import Path
import os
import shlex
import shutil
import subprocess
import yaml

COMFYUI_DIR = Path("/storage/ComfyUI")
MODEL_ROOT = Path("/app/models")
MANIFEST_PATH = Path("/storage/ComfyUI/hf-models.yaml")
MODEL_CATEGORY = "image"
FORCE_DOWNLOAD = False
COMFYUI_PORT = 6006
COMFYUI_ARGS = "--preview-method auto"
COMFYUI_PYTHON = None  # 例: "/storage/ComfyUI/.venv/bin/python"
PAPERSPACE_FQDN = os.environ.get("PAPERSPACE_FQDN")

# ComfyUI 側で扱う主なモデルディレクトリを定義します。
MODEL_TYPES = [
    "checkpoints",
    "loras",
    "vae",
    "clip",
    "clip_vision",
    "controlnet",
    "embeddings",
    "upscale_models",
    "text_encoders",
    "ipadapter",
    "diffusion_models",
    "unet",
]

# extra_model_paths.yaml は毎回この Notebook から再生成します。
EXTRA_MODEL_PATHS_FILE = COMFYUI_DIR / "extra_model_paths.yaml"
EXTRA_MODEL_PATHS_BACKUP = COMFYUI_DIR / "extra_model_paths.yaml.bak.paperspace-comfyui"

print(f"COMFYUI_DIR={COMFYUI_DIR}")
print(f"MODEL_ROOT={MODEL_ROOT}")
print(f"MANIFEST_PATH={MANIFEST_PATH}")
print(f"MODEL_CATEGORY={MODEL_CATEGORY}")
print(f"FORCE_DOWNLOAD={FORCE_DOWNLOAD}")
print(f"COMFYUI_PORT={COMFYUI_PORT}")
if PAPERSPACE_FQDN:
    print(f"PUBLIC_COMFYUI_URL=https://tensorboard-{PAPERSPACE_FQDN}")


In [ ]:
# コンテナ内コマンドと ComfyUI 配置を先に検証します。
assert shutil.which("hf"), "hf command not found"
assert COMFYUI_DIR.exists(), f"ComfyUI directory not found: {COMFYUI_DIR}"
assert (COMFYUI_DIR / "main.py").exists(), f"main.py not found under {COMFYUI_DIR}"

# モデル格納先はこのセルで先に用意しておきます。
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

python_version = subprocess.run(
    ["python", "-c", "import sys; print(sys.version.split()[0])"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

hf_process = subprocess.run(
    ["hf", "--version"],
    check=True,
    capture_output=True,
    text=True,
)
hf_version = hf_process.stdout.strip() or hf_process.stderr.strip()

print(f"python {python_version}")
print(hf_version)
print(f"MODEL_ROOT ready: {MODEL_ROOT}")


In [ ]:
# private repo に入るため HF_TOKEN を確認します。
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is not set. Configure it in Paperspace Secrets or environment variables.")

whoami = subprocess.run(
    ["hf", "whoami", "--token", HF_TOKEN],
    check=True,
    capture_output=True,
    text=True,
)
print(whoami.stdout.strip())


In [ ]:
# manifest を読み込み、選んだカテゴリのモデルだけを正規化します。
with MANIFEST_PATH.open("r", encoding="utf-8") as fh:
    manifest = yaml.safe_load(fh) or {}

if manifest.get("version") != 1:
    raise ValueError("Manifest version must be 1")

categories = manifest.get("categories")
if not isinstance(categories, dict):
    raise ValueError("Manifest must contain a categories mapping")

if MODEL_CATEGORY not in categories:
    raise KeyError(f"Category not found in manifest: {MODEL_CATEGORY}")

selected_models = categories[MODEL_CATEGORY]
if not isinstance(selected_models, list):
    raise ValueError(f"Category must contain a list: {MODEL_CATEGORY}")

normalized_models = []
for index, item in enumerate(selected_models, start=1):
    if not isinstance(item, dict):
        raise ValueError(f"Entry #{index} in {MODEL_CATEGORY} must be a mapping")

    repo = item.get("repo")
    source_path = item.get("source_path")
    target_subdir = item.get("target_subdir")
    revision = item.get("revision", "main")
    filename = item.get("filename") or Path(source_path).name

    if not repo or not source_path or not target_subdir:
        raise ValueError(f"Entry #{index} is missing repo, source_path, or target_subdir")
    if target_subdir not in MODEL_TYPES:
        raise ValueError(f"Unsupported target_subdir: {target_subdir}")

    normalized_models.append(
        {
            "repo": repo,
            "source_path": source_path,
            "target_subdir": target_subdir,
            "filename": filename,
            "revision": revision,
        }
    )

print(f"Loaded {len(normalized_models)} models for category '{MODEL_CATEGORY}'")
for item in normalized_models:
    print(f"- {item['target_subdir']}/{item['filename']} <- {item['repo']}:{item['source_path']}@{item['revision']}")


In [ ]:
# manifest の内容を順に hf download し、/app/models 配下へ配置します。
def hf_download(model):
    target_dir = MODEL_ROOT / model["target_subdir"]
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / model["filename"]

    # 既存ファイルを使うのが標準です。再取得したいときだけ FORCE_DOWNLOAD=True にします。
    if target_path.exists() and not FORCE_DOWNLOAD:
        print(f"skip: {target_path}")
        return target_path

    cmd = [
        "hf",
        "download",
        model["repo"],
        model["source_path"],
        "--repo-type",
        "model",
        "--revision",
        model["revision"],
        "--token",
        HF_TOKEN,
    ]
    if FORCE_DOWNLOAD:
        cmd.append("--force-download")

    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    downloaded_path = Path(result.stdout.strip().splitlines()[-1])

    if not downloaded_path.exists():
        raise FileNotFoundError(f"hf download did not return a valid path: {downloaded_path}")

    # HF キャッシュ上の実体を、ComfyUI が読む作業用ディレクトリへコピーします。
    shutil.copy2(downloaded_path, target_path)
    print(f"saved: {target_path}")
    return target_path

synced_paths = [hf_download(model) for model in normalized_models]
print(f"Synced {len(synced_paths)} model files to {MODEL_ROOT}")


In [ ]:
# /app/models を ComfyUI から見えるように extra_model_paths.yaml を再生成します。
extra_model_paths = {
    "hf_models": {
        "base_path": str(MODEL_ROOT),
    }
}
for model_type in MODEL_TYPES:
    extra_model_paths["hf_models"][model_type] = f"{model_type}/"

# 既存ファイルは退避してから上書きします。
if EXTRA_MODEL_PATHS_FILE.exists():
    shutil.copy2(EXTRA_MODEL_PATHS_FILE, EXTRA_MODEL_PATHS_BACKUP)
    print(f"Backed up existing file to {EXTRA_MODEL_PATHS_BACKUP}")

with EXTRA_MODEL_PATHS_FILE.open("w", encoding="utf-8") as fh:
    yaml.safe_dump(extra_model_paths, fh, sort_keys=False, allow_unicode=False)

print(f"Wrote {EXTRA_MODEL_PATHS_FILE}")
print(EXTRA_MODEL_PATHS_FILE.read_text(encoding="utf-8"))


In [ ]:
# ComfyUI 本体を起動します。Paperspace では 6006 を tensorboard-* URL で開く前提です。
# 永続ストレージ側の venv があれば優先して使います。
def resolve_comfyui_python():
    candidates = []
    if COMFYUI_PYTHON:
        candidates.append(Path(COMFYUI_PYTHON))
    candidates.extend(
        [
            COMFYUI_DIR / ".venv/bin/python",
            COMFYUI_DIR / "venv/bin/python",
        ]
    )

    for candidate in candidates:
        if candidate.exists():
            return candidate

    system_python = shutil.which("python")
    if system_python:
        return Path(system_python)

    raise FileNotFoundError("No usable Python executable found for ComfyUI")

comfyui_python = resolve_comfyui_python()
cmd = [
    str(comfyui_python),
    str(COMFYUI_DIR / "main.py"),
    "--listen",
    "0.0.0.0",
    "--port",
    str(COMFYUI_PORT),
]
if COMFYUI_ARGS.strip():
    cmd.extend(shlex.split(COMFYUI_ARGS))

print(f"Using python: {comfyui_python}")
print("$", shlex.join(cmd))
subprocess.run(cmd, cwd=COMFYUI_DIR, check=True)


In [ ]:
# 状況確認用のメモです。失敗時の確認にも使えます。
print("Notebook troubleshooting notes")
print(f"- Models root: {MODEL_ROOT}")
print(f"- extra_model_paths.yaml: {EXTRA_MODEL_PATHS_FILE}")
print(f"- Category: {MODEL_CATEGORY}")
print(f"- Paperspace FQDN: {PAPERSPACE_FQDN or '(not set)'}")

if PAPERSPACE_FQDN:
    print(f"- ComfyUI URL: https://tensorboard-{PAPERSPACE_FQDN}")

print("\nUseful commands:")
print(f"- ls -lah {MODEL_ROOT}")
print(f"- cat {EXTRA_MODEL_PATHS_FILE}")
print(f"- HF_TOKEN=*** hf whoami")
print(f"- cd {COMFYUI_DIR} && python main.py --help")
